### Group Information

| Field | Details |
|---|---|
| **Group ID** | Group J |
| **Student 1 Name** | Md. Nadim Srabon |
| **Student 1 ID** | 2023-1-60-047 |
| **Student 2 Name** | Pranay Saha |
| **Student 2 ID** | 2023-1-60-XXX |
| **Student 3 Name** | Redoanul Alam Mozumder |
| **Student 3 ID** | 2023-1-60-XXX |
| **Student 4 Name** | Aniruddha Manab Deb |
| **Student 4 ID** | 2023-1-60-XXX |
| **Notebook Type** | Dual-Stream Transformer (CNN + Landmarks) |
| **Dataset Source Name** | WLASL100 — Word-Level American Sign Language |
| **Dataset Source Link** | [github.com/dxli94/WLASL](https://github.com/dxli94/WLASL) |
| **Kaggle Dataset Path** | kaggle.com/datasets/nadimsrabon6/wlasl100-video-classification-dataset |
| **Submission Date** | DD Month 2026 |

---

<div style="background:linear-gradient(135deg,#993922,#F0E4E1);color:white;padding:20px 24px;border-radius:10px;">
<h2 style="margin:0 0 6px 0;">🤟 Sign Language Recognition</h2>
<p style="margin:0 0 10px 0;font-size:15px;opacity:0.95;">Dual-Stream Transformer Approach for Real-Time WLASL100 Word Recognition</p>
<p style="margin:0;font-size:13px;opacity:0.85;">📹 Dataset · 🧠 CNN + Landmark Streams · 🔄 Transformer Encoder · 📊 Evaluation · 🔍 Error Analysis</p>
</div>

---

### 📑 Table of Contents

1. [\*] [Global Configuration](#1) *(START HERE)*
2. [+] [Setup & Imports](#2)
3. [=] [Exploratory Data Analysis (EDA)](#3)
4. [W] [Data Preprocessing, Landmark Extraction & Augmentation](#4)
5. [T] [Training Infrastructure](#5)
6. [DS] [Dual-Stream Model: CNN-on-Hands + Landmark Stream → Fusion → Transformer Encoder](#6)
7. [ ] [Evaluation & Metrics (1-Epoch Sanity Check)](#7)
8. [S] [Robustness Analysis](#8) *(deferred — flagged for after full training)*
9. [?] [Error Analysis](#9) *(deferred — flagged for after full training)*


## 1. [\*] Global Configuration `(START HERE)`

All hyperparameters, paths, seeds, and toggles live in this single cell. **No hardcoded values should
appear anywhere else in the notebook** — every other cell reads from `CFG`.

This run is deliberately set to **`NUM_EPOCHS = 1`** as a sanity/timing check. Once timing and shapes are
confirmed, increase `NUM_EPOCHS` and revisit `LEARNING_RATE` / `BATCH_SIZE` in a hyperparameter sweep.


In [ ]:

import os
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    # ── Paths ────────────────────────────────────────────────────────
    DATA_ROOT: str        = "/kaggle/input/datasets/nadimsrabon6/wlasl100-video-classification-dataset/WLASL100/WLASL100"
    VIDEOS_DIR: str        = field(init=False)
    SPLITS_DIR: str        = field(init=False)
    LABEL_MAP_PATH: str    = field(init=False)
    OUTPUT_DIR: str        = "/kaggle/working/outputs"
    CHECKPOINT_DIR: str    = field(init=False)

    # ── Reproducibility ──────────────────────────────────────────────
    SEED: int              = 42

    # ── Video / Frame sampling ───────────────────────────────────────
    NUM_FRAMES: int        = 16          # T = 16, per proposed methodology
    FRAME_SIZE: int        = 224         # resize for hand-patch CNN input
    HAND_CROP_SIZE: int    = 112         # cropped hand patch resolution

    # ── MediaPipe ─────────────────────────────────────────────────────
    MP_MIN_DETECTION_CONF: float = 0.5
    MP_MIN_TRACKING_CONF: float  = 0.5
    LANDMARK_DIM: int      = field(init=False)   # computed below

    # ── Stream 1: CNN-on-hands ────────────────────────────────────────
    CNN_BACKBONE: str      = "resnet18"  # 'resnet18' or 'efficientnet_b0'
    CNN_PRETRAINED: bool   = True
    CNN_EMBED_DIM: int     = 256         # projected embedding per hand-CNN

    # ── Stream 2: Landmark geometry ───────────────────────────────────
    LANDMARK_EMBED_DIM: int = 256        # projected embedding for landmark stream

    # ── Fusion + Transformer encoder ──────────────────────────────────
    FUSION_DIM: int        = 256         # common dim after fusion projection
    TRANSFORMER_LAYERS: int = 2          # shallow: WLASL100 is a small dataset
    TRANSFORMER_HEADS: int  = 4
    TRANSFORMER_FF_DIM: int = 512
    TRANSFORMER_DROPOUT: float = 0.1

    # ── Classification head ──────────────────────────────────────────
    NUM_CLASSES: int       = 100         # WLASL100

    # ── Training ──────────────────────────────────────────────────────
    BATCH_SIZE: int        = 8           # conservative for first sanity run
    NUM_EPOCHS: int        = 1           # === SANITY RUN: confirm timing before scaling ===
    LEARNING_RATE: float   = 3e-4
    WEIGHT_DECAY: float    = 1e-4
    WARMUP_STEPS: int      = 100
    GRAD_CLIP_NORM: float  = 1.0
    USE_AMP: bool          = True        # mixed precision (torch.cuda.amp)

    # ── Dataloader ─────────────────────────────────────────────────────
    NUM_WORKERS: int       = 0
    PIN_MEMORY: bool       = True

    # ── Device ───────────────────────────────────────────────────────
    DEVICE: str            = "cuda"      # falls back to cpu if unavailable, see Section 2

    # ── Accuracy gate (per proposed methodology, Section 5) ───────────
    SENTENCE_LEVEL_GATE_ACC: float = 0.70   # not used this run; documented for future notebooks

    def __post_init__(self):
        self.VIDEOS_DIR     = os.path.join(self.DATA_ROOT, "videos")
        self.SPLITS_DIR     = os.path.join(self.DATA_ROOT, "splits")
        self.LABEL_MAP_PATH = os.path.join(self.DATA_ROOT, "label_map.json")
        self.CHECKPOINT_DIR = os.path.join(self.OUTPUT_DIR, "checkpoints")
        # 2 hands * 21 landmarks * 3 coords + pose(33*3) + face_subset(~20*3)
        self.LANDMARK_DIM = (2 * 21 * 3) + (33 * 3) + (20 * 3)

CFG = Config()
os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
os.makedirs(CFG.CHECKPOINT_DIR, exist_ok=True)

print(f"Landmark feature dim (Stream 2 input): {CFG.LANDMARK_DIM}")
print(f"Sanity run epochs: {CFG.NUM_EPOCHS}  |  Batch size: {CFG.BATCH_SIZE}  |  T (frames): {CFG.NUM_FRAMES}")



## 2. [+] Setup & Imports

Install any packages not pre-installed on Kaggle, import all libraries, confirm GPU availability, and
print framework versions. `mediapipe` is not part of the default Kaggle image and must be installed here.


In [ ]:

# Kaggle's default image does not ship mediapipe — install it
!pip install -q mediapipe==0.10.14


In [ ]:

import sys, time, json, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import mediapipe as mp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

import torchvision
import torchvision.models as tvm
from torchvision import transforms

# ── Reproducibility ──────────────────────────────────────────────────
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG.SEED)

# ── Device check ──────────────────────────────────────────────────────
if torch.cuda.is_available():
    CFG.DEVICE = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
else:
    CFG.DEVICE = "cpu"
    gpu_name = "N/A — running on CPU, expect this sanity run to be slow"

device = torch.device(CFG.DEVICE)

print(f"PyTorch version      : {torch.__version__}")
print(f"Torchvision version   : {torchvision.__version__}")
print(f"OpenCV version        : {cv2.__version__}")
print(f"MediaPipe version      : {mp.__version__}")
print(f"CUDA available         : {torch.cuda.is_available()}")
print(f"Device in use          : {CFG.DEVICE}  ({gpu_name})")



## 3. [=] Exploratory Data Analysis (EDA)

Before committing to `NUM_FRAMES = 16`, we need to actually measure the dataset's clip-length distribution
— per the proposal discussion, clip trimming tightness is **unknown/variable**, so this section confirms
whether 16-frame uniform sampling is a reasonable choice or needs revisiting.

We examine: class distribution (balance), video duration / frame-count spread, resolution consistency,
and a visual sample grid per class.


In [ ]:
# ── Load splits and label map ────────────────────────────────────────
train_df = pd.read_csv(os.path.join(CFG.SPLITS_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(CFG.SPLITS_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(CFG.SPLITS_DIR, "test.csv"))

with open(CFG.LABEL_MAP_PATH) as f:
    label_map = json.load(f)          # {gloss: int_id}
inv_label_map = {v: k for k, v in label_map.items()}

# Actual CSV columns: "video" (already includes .mp4) and "label" (already an int class id)
video_col = "video"
label_col = "label"

print(f"Train samples: {len(train_df)}")
print(f"Val samples  : {len(val_df)}")
print(f"Test samples : {len(test_df)}")
print(f"Total classes: {len(label_map)}")
print()
print("train.csv columns:", list(train_df.columns))
train_df.head()

In [ ]:

# ── Class distribution ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, df, name in zip(axes, [train_df, val_df, test_df], ["Train", "Val", "Test"]):
    counts = df["gloss"].value_counts() if "gloss" in df.columns else df.iloc[:, 1].value_counts()
    ax.hist(counts.values, bins=20, color="#1565C0", edgecolor="white")
    ax.set_title(f"{name}: samples-per-class distribution")
    ax.set_xlabel("samples in class")
    ax.set_ylabel("number of classes")
plt.tight_layout()
plt.savefig(os.path.join(CFG.OUTPUT_DIR, "class_distribution.png"), dpi=120)
plt.show()

print(f"Train — min/max/mean samples per class: "
      f"{counts.min()} / {counts.max()} / {counts.mean():.1f}")


In [ ]:
# ── Video duration / frame-count spread (this determines our T=16 sampling strategy) ──
def probe_video(path):
    cap = cv2.VideoCapture(path)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps      = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return n_frames, fps, width, height

N_PROBE = min(200, len(train_df))
probe_rows = train_df.sample(N_PROBE, random_state=CFG.SEED)

probe_results = []
missing_count = 0
for _, row in probe_rows.iterrows():
    # video_col already contains ".mp4" — do NOT append it again
    vid_path = os.path.join(CFG.VIDEOS_DIR, row[video_col])
    if os.path.exists(vid_path):
        n, fps, w, h = probe_video(vid_path)
        probe_results.append({"n_frames": n, "fps": fps, "width": w, "height": h,
                               "duration_s": n / fps if fps > 0 else None})
    else:
        missing_count += 1

print(f"Probed: {len(probe_results)} found, {missing_count} missing out of {N_PROBE}")

probe_df = pd.DataFrame(probe_results)
if len(probe_df) == 0:
    raise RuntimeError(
        "No video files found — check CFG.VIDEOS_DIR path and confirm filenames "
        "in train.csv match actual files in the videos/ folder."
    )

print(probe_df.describe())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(probe_df["n_frames"], bins=25, color="#00695C", edgecolor="white")
axes[0].axvline(CFG.NUM_FRAMES, color="red", linestyle="--", label=f"T={CFG.NUM_FRAMES}")
axes[0].set_title("Frame count per clip")
axes[0].legend()
axes[1].hist(probe_df["duration_s"], bins=25, color="#6A1B9A", edgecolor="white")
axes[1].set_title("Clip duration (seconds)")
plt.tight_layout()
plt.savefig(os.path.join(CFG.OUTPUT_DIR, "duration_distribution.png"), dpi=120)
plt.show()


**Interpretation note (fill in after running):** if `n_frames` varies widely (e.g., some clips are 20
frames, others 90+), uniform sampling to `T=16` is doing meaningfully different things per clip —
short clips get near-complete coverage, long clips get sparse sampling. Revisit `NUM_FRAMES` or consider
a duration-aware sampling strategy if the spread is large. This is exactly the kind of "variable trimming"
risk flagged before building this notebook — confirm here rather than assume.


In [ ]:

# ── Resolution consistency check ──────────────────────────────────────
print("Resolution value counts (width x height):")
print(probe_df.groupby(["width", "height"]).size().sort_values(ascending=False))


In [ ]:
# ── Visual sample grid: one frame from N random classes ────────────────
sample_labels = random.sample(sorted(train_df[label_col].unique().tolist()), min(8, train_df[label_col].nunique()))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, lbl in zip(axes.flat, sample_labels):
    rows = train_df[train_df[label_col] == lbl]
    if len(rows) == 0:
        continue
    vid_path = os.path.join(CFG.VIDEOS_DIR, rows.iloc[0][video_col])
    cap = cv2.VideoCapture(vid_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT) // 2)))
    ret, frame = cap.read()
    cap.release()
    if ret:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ax.imshow(frame)
    gloss_name = inv_label_map.get(lbl, str(lbl))
    ax.set_title(gloss_name, fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(CFG.OUTPUT_DIR, "sample_grid.png"), dpi=120)
plt.show()


## 4. [W] Data Preprocessing, Landmark Extraction & Augmentation

This is the core of the **dual-stream** input pipeline, run live with MediaPipe Holistic per the
methodology (Section 1, Stage 1–2):

- **Frame sampling:** uniform `T=16` sampling across each clip (revisit per EDA findings above).
- **MediaPipe Holistic** runs per sampled frame, returning hand bounding boxes (for Stream 1 crops) and
  hand/pose/face keypoints (for Stream 2).
- **Stream 1 input:** left + right hand patches, resized to `HAND_CROP_SIZE`, ImageNet-normalized.
- **Stream 2 input:** keypoints normalized relative to shoulder width (signer-scale invariance), flattened
  to a per-frame vector of dimension `CFG.LANDMARK_DIM`.
- **Missing-hand fallback:** if MediaPipe fails to detect a hand in a frame (common — occlusion, motion
  blur), we zero-pad that hand's crop and landmarks rather than dropping the frame, to keep `T` fixed.
- **Augmentation:** horizontal flip (mirrors left/right hand labels — must be handled carefully for sign
  language, see note below), slight rotation, and brightness jitter on hand crops only — landmarks are
  re-derived from frames, not synthetically perturbed, to keep geometry physically valid.

> **Caution flagged for later:** horizontal flipping a sign-language clip can change handedness, and for
> ASL specifically may invert meaning for handedness-sensitive signs. We apply it cautiously, document it
> here, and recommend an ablation (with/without flip) once full-scale training begins.


In [ ]:

# ── MediaPipe Holistic setup ──────────────────────────────────────────
mp_holistic = mp.solutions.holistic

# Indices for a reduced facial subset — eyebrows + mouth (non-manual markers)
FACE_SUBSET_IDX = [
    # eyebrows (left+right approx)
    70, 63, 105, 66, 107, 336, 296, 334, 293, 300,
    # mouth outer ring
    61, 291, 0, 17, 78, 308, 13, 14, 87, 317
]  # 20 points, matches CFG.LANDMARK_DIM calc in Section 1

def extract_holistic_features(frame_rgb, holistic_model):
    '''
    Runs MediaPipe Holistic on a single RGB frame.
    Returns:
        left_hand_box, right_hand_box : (x_min,y_min,x_max,y_max) in pixel coords, or None
        landmark_vec : flattened np.array of shape (CFG.LANDMARK_DIM,)
    '''
    h, w, _ = frame_rgb.shape
    results = holistic_model.process(frame_rgb)

    def bbox_from_landmarks(landmarks, pad=0.25):
        if landmarks is None:
            return None
        xs = [pt.x * w for pt in landmarks.landmark]
        ys = [pt.y * h for pt in landmarks.landmark]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        pad_x = (x_max - x_min) * pad
        pad_y = (y_max - y_min) * pad
        return (max(0, int(x_min - pad_x)), max(0, int(y_min - pad_y)),
                min(w, int(x_max + pad_x)), min(h, int(y_max + pad_y)))

    left_box  = bbox_from_landmarks(results.left_hand_landmarks)
    right_box = bbox_from_landmarks(results.right_hand_landmarks)

    # ── Landmark vector (Stream 2) ──
    def flat_or_zero(landmarks, n_points, indices=None):
        if landmarks is None:
            return np.zeros(n_points * 3, dtype=np.float32)
        pts = landmarks.landmark
        if indices is not None:
            pts = [pts[i] for i in indices]
        return np.array([[p.x, p.y, p.z] for p in pts], dtype=np.float32).flatten()

    left_hand_vec  = flat_or_zero(results.left_hand_landmarks, 21)
    right_hand_vec = flat_or_zero(results.right_hand_landmarks, 21)
    pose_vec       = flat_or_zero(results.pose_landmarks, 33)
    face_vec       = flat_or_zero(results.face_landmarks, 20, FACE_SUBSET_IDX)

    landmark_vec = np.concatenate([left_hand_vec, right_hand_vec, pose_vec, face_vec])

    # ── Shoulder-width normalization (signer-scale invariance) ──
    if results.pose_landmarks is not None:
        lm = results.pose_landmarks.landmark
        l_sh, r_sh = lm[11], lm[12]   # left_shoulder, right_shoulder (MediaPipe pose indices)
        shoulder_width = np.linalg.norm([l_sh.x - r_sh.x, l_sh.y - r_sh.y]) + 1e-6
        landmark_vec = landmark_vec / shoulder_width

    return left_box, right_box, landmark_vec.astype(np.float32)


def crop_hand(frame_rgb, box, size):
    if box is None:
        return np.zeros((size, size, 3), dtype=np.uint8)
    x1, y1, x2, y2 = box
    crop = frame_rgb[y1:y2, x1:x2]
    if crop.size == 0:
        return np.zeros((size, size, 3), dtype=np.uint8)
    return cv2.resize(crop, (size, size))


def sample_frame_indices(n_total_frames, n_sample):
    '''Uniform sampling across the clip. Revisit if EDA shows highly variable clip lengths.'''
    if n_total_frames <= n_sample:
        idxs = list(range(n_total_frames)) + [n_total_frames - 1] * (n_sample - n_total_frames)
    else:
        idxs = np.linspace(0, n_total_frames - 1, n_sample).astype(int).tolist()
    return idxs


In [ ]:

# ── ImageNet normalization for hand-patch CNN stream ───────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

hand_transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(p=0.0),   # see handedness caution note above — disabled by default
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

hand_transform_eval = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


In [ ]:

class WLASLDualStreamDataset(Dataset):
    '''
    Loads a clip, samples T frames, runs MediaPipe Holistic per frame,
    and returns:
        left_hand_patches, right_hand_patches : (T, 3, H, W) each
        landmark_seq                           : (T, LANDMARK_DIM)
        label                                  : int
    '''
    def __init__(self, df, label_map, cfg, split="train"):
        self.df = df.reset_index(drop=True)
        self.label_map = label_map
        self.cfg = cfg
        self.split = split
        self.video_col = "video"   # already includes ".mp4"
        self.label_col = "label"   # already an integer class id — no label_map lookup needed
        self.transform = hand_transform_train if split == "train" else hand_transform_eval

        # One Holistic instance reused per worker (created lazily — MediaPipe objects are not fork-safe)
        self._holistic = None

    def _get_holistic(self):
        if self._holistic is None:
            self._holistic = mp_holistic.Holistic(
                min_detection_confidence=self.cfg.MP_MIN_DETECTION_CONF,
                min_tracking_confidence=self.cfg.MP_MIN_TRACKING_CONF,
                model_complexity=1,
            )
        return self._holistic

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_path = os.path.join(self.cfg.VIDEOS_DIR, row[self.video_col])
        label = int(row[self.label_col])

        cap = cv2.VideoCapture(video_path)
        n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_idxs = sample_frame_indices(max(n_frames, 1), self.cfg.NUM_FRAMES)

        holistic = self._get_holistic()
        left_patches, right_patches, landmark_seq = [], [], []

        frame_idx_set = set(frame_idxs)
        frames_cache = {}
        current = 0
        while cap.isOpened() and len(frames_cache) < len(frame_idx_set):
            ret, frame = cap.read()
            if not ret:
                break
            if current in frame_idx_set:
                frames_cache[current] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            current += 1
        cap.release()

        last_valid = np.zeros((self.cfg.HAND_CROP_SIZE, self.cfg.HAND_CROP_SIZE, 3), dtype=np.uint8)
        for fi in frame_idxs:
            frame_rgb = frames_cache.get(fi, last_valid)
            l_box, r_box, lm_vec = extract_holistic_features(frame_rgb, holistic)
            left_patches.append(self.transform(crop_hand(frame_rgb, l_box, self.cfg.HAND_CROP_SIZE)))
            right_patches.append(self.transform(crop_hand(frame_rgb, r_box, self.cfg.HAND_CROP_SIZE)))
            landmark_seq.append(lm_vec)

        left_patches  = torch.stack(left_patches)             # (T, 3, H, W)
        right_patches = torch.stack(right_patches)            # (T, 3, H, W)
        landmark_seq  = torch.from_numpy(np.stack(landmark_seq))  # (T, LANDMARK_DIM)

        return {
            "left_hand":  left_patches,
            "right_hand": right_patches,
            "landmarks":  landmark_seq,
            "label": torch.tensor(label, dtype=torch.long),
        }


In [ ]:

# ── Quick shape sanity check on a single sample before building full DataLoaders ──
_sanity_ds = WLASLDualStreamDataset(train_df.head(2), label_map, CFG, split="train")
_sample = _sanity_ds[0]
print("left_hand  :", _sample["left_hand"].shape)
print("right_hand :", _sample["right_hand"].shape)
print("landmarks  :", _sample["landmarks"].shape, "(expected dim:", CFG.LANDMARK_DIM, ")")
print("label      :", _sample["label"].item())


In [ ]:
# ── Full datasets and dataloaders ──────────────────────────────────────
# NOTE: num_workers must stay 0. MediaPipe Holistic uses a C++ backend that is not
# fork-safe — PyTorch's default multiprocessing start method (fork, on Linux) corrupts
# MediaPipe's internal state across worker processes, producing the
# "malloc(): unaligned tcache chunk detected" crash and silent worker death.
# Single-process loading is slower but correct. Pre-extracting landmarks/crops to disk
# in a separate offline pass (instead of running MediaPipe live in the DataLoader) is
# the right fix once we move past this 1-epoch sanity check.

train_dataset = WLASLDualStreamDataset(train_df, label_map, CFG, split="train")
val_dataset   = WLASLDualStreamDataset(val_df,   label_map, CFG, split="val")
test_dataset  = WLASLDualStreamDataset(test_df,  label_map, CFG, split="test")

train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True,
                           num_workers=0, pin_memory=CFG.PIN_MEMORY)
val_loader   = DataLoader(val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False,
                           num_workers=0, pin_memory=CFG.PIN_MEMORY)
test_loader  = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False,
                           num_workers=0, pin_memory=CFG.PIN_MEMORY)

print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}  |  Test batches: {len(test_loader)}")


## 5. [T] Training Infrastructure

A generic training loop supporting: loss function, AdamW optimizer, cosine LR scheduling with warmup,
gradient clipping, mixed-precision training, and best-checkpoint saving. This loop is model-agnostic and
will be reused unchanged when we later add the sentence-level CTC head (Section 5 of the proposed
methodology) — only the loss/decoding stage changes downstream.


In [ ]:

def build_optimizer_and_scheduler(model, cfg, num_training_steps):
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)

    def lr_lambda(step):
        if step < cfg.WARMUP_STEPS:
            return step / max(1, cfg.WARMUP_STEPS)
        progress = (step - cfg.WARMUP_STEPS) / max(1, num_training_steps - cfg.WARMUP_STEPS)
        return 0.5 * (1 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


def train_one_epoch(model, loader, optimizer, scheduler, scaler, criterion, cfg, device):
    model.train()
    running_loss, all_preds, all_labels = 0.0, [], []
    t0 = time.time()

    for step, batch in enumerate(loader):
        left  = batch["left_hand"].to(device)
        right = batch["right_hand"].to(device)
        lms   = batch["landmarks"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        with autocast(enabled=cfg.USE_AMP):
            logits = model(left, right, lms)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()
        all_preds.extend(logits.argmax(dim=1).detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

        if step % max(1, len(loader)//5) == 0:
            print(f"  step {step:>4d}/{len(loader)}  loss={loss.item():.4f}")

    elapsed = time.time() - t0
    acc = accuracy_score(all_labels, all_preds)
    return running_loss / len(loader), acc, elapsed


@torch.no_grad()
def evaluate(model, loader, criterion, cfg, device):
    model.eval()
    running_loss, all_preds, all_labels = 0.0, [], []

    for batch in loader:
        left  = batch["left_hand"].to(device)
        right = batch["right_hand"].to(device)
        lms   = batch["landmarks"].to(device)
        labels = batch["label"].to(device)

        with autocast(enabled=cfg.USE_AMP):
            logits = model(left, right, lms)
            loss = criterion(logits, labels)

        running_loss += loss.item()
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return running_loss / len(loader), acc, all_preds, all_labels


def save_checkpoint(model, optimizer, epoch, val_acc, cfg, is_best=False):
    path = os.path.join(cfg.CHECKPOINT_DIR, "last_checkpoint.pt")
    torch.save({
        "epoch": epoch, "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(), "val_acc": val_acc,
    }, path)
    if is_best:
        best_path = os.path.join(cfg.CHECKPOINT_DIR, "best_checkpoint.pt")
        torch.save({
            "epoch": epoch, "model_state": model.state_dict(), "val_acc": val_acc,
        }, best_path)
        print(f"  ✓ New best checkpoint saved (val_acc={val_acc:.4f})")



## 6. [DS] Dual-Stream Model: CNN-on-Hands + Landmark Geometry → Fusion → Transformer Encoder

Implements the architecture from the proposed methodology exactly:

- **Stream 1 (appearance):** per-frame left + right hand crops → shared CNN backbone (ImageNet-pretrained
  `ResNet-18`) → concatenated hand embeddings → linear projection to `CNN_EMBED_DIM`.
- **Stream 2 (geometry):** per-frame normalized landmark vector → linear projection to `LANDMARK_EMBED_DIM`.
- **Fusion:** concatenate Stream 1 + Stream 2 **per frame**, project to common `FUSION_DIM` — fusion
  happens *before* temporal modeling, not after, so self-attention can relate handshape to spatial position.
- **Temporal encoder:** standard Transformer encoder (sinusoidal positional encoding, `TRANSFORMER_LAYERS`
  layers) over the `T=16` fused sequence.
- **Head:** mean-pool over time, then softmax classification over `NUM_CLASSES` (word-level only this run
  — sentence-level CTC head is gated behind the accuracy threshold and implemented in a later notebook).


In [ ]:

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, dim, max_len=64):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-np.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))   # (1, max_len, dim)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class HandCNNEncoder(nn.Module):
    '''Shared CNN backbone applied to both hand streams, frame-by-frame.'''
    def __init__(self, backbone="resnet18", pretrained=True, embed_dim=256):
        super().__init__()
        if backbone == "resnet18":
            net = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
            feat_dim = net.fc.in_features
            net.fc = nn.Identity()
            self.backbone = net
        elif backbone == "efficientnet_b0":
            net = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
            feat_dim = net.classifier[1].in_features
            net.classifier = nn.Identity()
            self.backbone = net
        else:
            raise ValueError(f"Unsupported backbone: {backbone}")

        self.proj = nn.Linear(feat_dim, embed_dim)

    def forward(self, x):
        # x: (B, T, 3, H, W) -> flatten time into batch for the CNN, then reshape back
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        feats = self.backbone(x)              # (B*T, feat_dim)
        feats = self.proj(feats)              # (B*T, embed_dim)
        return feats.view(B, T, -1)            # (B, T, embed_dim)


class DualStreamTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        # Stream 1: shared CNN for both hands
        self.hand_cnn = HandCNNEncoder(cfg.CNN_BACKBONE, cfg.CNN_PRETRAINED, cfg.CNN_EMBED_DIM)
        self.hand_fuse_proj = nn.Linear(cfg.CNN_EMBED_DIM * 2, cfg.CNN_EMBED_DIM)  # left+right -> 1 vec

        # Stream 2: landmark geometry
        self.landmark_proj = nn.Sequential(
            nn.Linear(cfg.LANDMARK_DIM, cfg.LANDMARK_EMBED_DIM),
            nn.ReLU(),
            nn.LayerNorm(cfg.LANDMARK_EMBED_DIM),
        )

        # Fusion: concat(Stream1, Stream2) -> common dim, BEFORE temporal modeling
        self.fusion_proj = nn.Linear(cfg.CNN_EMBED_DIM + cfg.LANDMARK_EMBED_DIM, cfg.FUSION_DIM)

        # Temporal: Transformer encoder over T frames
        self.pos_encoding = SinusoidalPositionalEncoding(cfg.FUSION_DIM, max_len=cfg.NUM_FRAMES + 4)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.FUSION_DIM, nhead=cfg.TRANSFORMER_HEADS,
            dim_feedforward=cfg.TRANSFORMER_FF_DIM, dropout=cfg.TRANSFORMER_DROPOUT,
            batch_first=True, activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg.TRANSFORMER_LAYERS)

        # Word-level classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(cfg.FUSION_DIM),
            nn.Linear(cfg.FUSION_DIM, cfg.FUSION_DIM // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(cfg.FUSION_DIM // 2, cfg.NUM_CLASSES),
        )

    def forward(self, left_hand, right_hand, landmarks):
        # Stream 1
        left_feat  = self.hand_cnn(left_hand)     # (B, T, CNN_EMBED_DIM)
        right_feat = self.hand_cnn(right_hand)    # (B, T, CNN_EMBED_DIM)
        hand_feat  = self.hand_fuse_proj(torch.cat([left_feat, right_feat], dim=-1))  # (B, T, CNN_EMBED_DIM)

        # Stream 2
        lm_feat = self.landmark_proj(landmarks)   # (B, T, LANDMARK_EMBED_DIM)

        # Per-frame fusion BEFORE temporal modeling
        fused = self.fusion_proj(torch.cat([hand_feat, lm_feat], dim=-1))  # (B, T, FUSION_DIM)
        fused = self.pos_encoding(fused)

        # Temporal encoder
        encoded = self.transformer(fused)         # (B, T, FUSION_DIM)

        # Mean-pool over time, classify
        pooled = encoded.mean(dim=1)               # (B, FUSION_DIM)
        logits = self.classifier(pooled)            # (B, NUM_CLASSES)
        return logits


In [ ]:

# ── Instantiate model, report trainable parameters ──────────────────
model = DualStreamTransformer(CFG).to(device)

n_total     = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {n_total:,}")
print(f"Trainable parameters : {n_trainable:,}")


In [ ]:

# ── Forward-pass shape sanity check before full training ──────────────
model.eval()
with torch.no_grad():
    _batch = next(iter(train_loader))
    _logits = model(
        _batch["left_hand"].to(device),
        _batch["right_hand"].to(device),
        _batch["landmarks"].to(device),
    )
print("Logits shape:", _logits.shape, "(expected: [BATCH_SIZE, NUM_CLASSES])")



### 6.1 — Run Training (`NUM_EPOCHS = 1`, sanity / timing check)

This is the actual training run for this notebook pass. We are **not** hyperparameter tuning yet — the
sole purpose of this run is to confirm (a) the pipeline executes end-to-end without shape/runtime errors,
and (b) measure wall-clock time per epoch on the Kaggle T4/P100, which determines what's feasible for the
full multi-epoch run.


In [ ]:

criterion = nn.CrossEntropyLoss()
optimizer, scheduler = build_optimizer_and_scheduler(model, CFG, num_training_steps=len(train_loader) * CFG.NUM_EPOCHS)
scaler = GradScaler(enabled=CFG.USE_AMP)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "epoch_time_s": []}
best_val_acc = 0.0

for epoch in range(1, CFG.NUM_EPOCHS + 1):
    print(f"\n=== Epoch {epoch}/{CFG.NUM_EPOCHS} ===")

    train_loss, train_acc, epoch_time = train_one_epoch(
        model, train_loader, optimizer, scheduler, scaler, criterion, CFG, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, CFG, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["epoch_time_s"].append(epoch_time)

    is_best = val_acc > best_val_acc
    best_val_acc = max(best_val_acc, val_acc)
    save_checkpoint(model, optimizer, epoch, val_acc, CFG, is_best=is_best)

    print(f"  train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  "
          f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  "
          f"epoch_time={epoch_time:.1f}s")

print(f"\n{'='*60}")
print(f"SANITY RUN COMPLETE — {CFG.NUM_EPOCHS} epoch(s)")
print(f"Time per epoch: {history['epoch_time_s'][0]:.1f}s "
      f"(~{history['epoch_time_s'][0]/60:.1f} min)")
print(f"Estimated time for 30 epochs: {history['epoch_time_s'][0]*30/60:.1f} min")
print(f"{'='*60}")



> **Next step after this run:** look at `epoch_time_s` above and decide a realistic `NUM_EPOCHS` for the
> full run given Kaggle's session time limit. Then revisit `LEARNING_RATE`, `BATCH_SIZE`, and
> `TRANSFORMER_LAYERS` as the actual hyperparameter sweep — this run is deliberately conservative on all
> three so the timing number reflects a safe floor, not a best case.



## 7. [ ] Evaluation & Metrics (1-Epoch Sanity Check)

Standard metrics on the test set: accuracy, precision, recall, F1 (macro & weighted), confusion matrix,
and per-batch inference time. **At 1 epoch these numbers are not meaningful for judging model quality** —
they exist purely to confirm the evaluation code path works correctly before the full run.


In [ ]:

test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, CFG, device)

precision_macro = precision_score(test_labels, test_preds, average="macro", zero_division=0)
recall_macro    = recall_score(test_labels, test_preds, average="macro", zero_division=0)
f1_macro        = f1_score(test_labels, test_preds, average="macro", zero_division=0)
f1_weighted     = f1_score(test_labels, test_preds, average="weighted", zero_division=0)

print("=== Test Set Metrics (1-epoch sanity check — not indicative of final performance) ===")
print(f"Accuracy          : {test_acc:.4f}")
print(f"Precision (macro)  : {precision_macro:.4f}")
print(f"Recall (macro)     : {recall_macro:.4f}")
print(f"F1 (macro)         : {f1_macro:.4f}")
print(f"F1 (weighted)      : {f1_weighted:.4f}")


In [ ]:

# ── Training curves (will be flat/noisy at 1 epoch — meaningful once multi-epoch) ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], marker="o", label="train")
axes[0].plot(history["val_loss"], marker="o", label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(history["train_acc"], marker="o", label="train")
axes[1].plot(history["val_acc"], marker="o", label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(CFG.OUTPUT_DIR, "training_curves.png"), dpi=120)
plt.show()


In [ ]:

# ── Confusion matrix (will look near-random at 1 epoch — that is expected) ──
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap="Blues", cbar=True)
plt.title("Confusion Matrix — 1-Epoch Sanity Check")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(CFG.OUTPUT_DIR, "confusion_matrix.png"), dpi=120)
plt.show()


In [ ]:

# ── Per-batch inference time (useful now — informs real-time latency budget) ──
model.eval()
inference_times = []
with torch.no_grad():
    for batch in test_loader:
        left  = batch["left_hand"].to(device)
        right = batch["right_hand"].to(device)
        lms   = batch["landmarks"].to(device)
        t0 = time.time()
        with autocast(enabled=CFG.USE_AMP):
            _ = model(left, right, lms)
        torch.cuda.synchronize() if CFG.DEVICE == "cuda" else None
        inference_times.append(time.time() - t0)

inference_times = np.array(inference_times)
per_sample_ms = (inference_times.mean() / CFG.BATCH_SIZE) * 1000
print(f"Avg batch inference time: {inference_times.mean()*1000:.1f} ms  (batch size {CFG.BATCH_SIZE})")
print(f"Approx. per-sample inference time: {per_sample_ms:.1f} ms")
print(f"(Compare against the ~250ms end-to-end latency budget from the proposed methodology —")
print(f" this number covers model forward pass only, not capture/MediaPipe/decode/TTS.)")



## 8. [S] Robustness Analysis

**Deferred to after full-scale training.** At 1 epoch the model has not learned enough for robustness
results to be informative — testing distribution shifts (Gaussian noise, brightness/contrast, JPEG
compression) on an undertrained model would mostly measure noise on noise. This section is structurally
present per the notebook standard, and will be filled in once `NUM_EPOCHS` is increased and the model
reaches a meaningful baseline accuracy.

**Planned for later:** Gaussian noise injection, brightness/contrast shifts, and JPEG compression — applied
identically to both hand-crop and landmark-extraction stages, with performance-degradation curves plotted
per shift type.



## 9. [?] Error Analysis

**Deferred to after full-scale training**, for the same reason as Section 8 — at 1 epoch, nearly every
prediction is effectively random, so misclassification patterns would not reflect genuine model behavior.

**Planned for later:** identify top-$k$ misclassified samples, visualize predicted vs. true labels, and —
specific to this dual-stream architecture — inspect whether errors correlate with frames where MediaPipe
failed to detect a hand (i.e., whether the zero-padding fallback from Section 4 is a meaningful error
source). This last check is unique to this pipeline and worth prioritizing early in error analysis once
real training begins.
